# Governance Dashboard — QuickSight Analysis for Inference Monitoring

This notebook programmatically creates a complete QuickSight dashboard — no manual UI steps.

**What it creates (all via API):**
- Athena data source in QuickSight
- Dataset from `inference_responses` table (auto-discovered columns + calculated fields)
- Analysis with 6 visuals (Definition mode)
- Published dashboard

**Prerequisites:**
- QuickSight Enterprise subscription
- Data in `inference_responses` Athena table
- IAM permissions for QuickSight, Athena, S3

## 1. Setup & Configuration

In [ ]:
import sys, os, json, time, boto3
from pathlib import Path
from botocore.exceptions import ClientError
from dotenv import load_dotenv

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))
load_dotenv(project_root / '.env')

from src.config.config import (
    ATHENA_DATABASE, AWS_DEFAULT_REGION, ATHENA_OUTPUT_S3, DATA_S3_BUCKET
)

sts = boto3.client('sts')
quicksight = boto3.client('quicksight', region_name=AWS_DEFAULT_REGION)
athena = boto3.client('athena', region_name=AWS_DEFAULT_REGION)

ACCOUNT_ID = sts.get_caller_identity()['Account']
REGION = AWS_DEFAULT_REGION

DATASOURCE_ID = 'fraud-governance-athena-datasource'
DATASET_ID    = 'fraud-governance-inference-dataset'
ANALYSIS_ID   = 'fraud-governance-analysis'
DASHBOARD_ID  = 'fraud-governance-dashboard'

print(f'Account:  {ACCOUNT_ID}')
print(f'Region:   {REGION}')
print(f'Database: {ATHENA_DATABASE}')

## 2. Verify QuickSight Subscription

In [ ]:
try:
    qs = quicksight.describe_account_settings(AwsAccountId=ACCOUNT_ID)
    edition = qs['AccountSettings'].get('Edition', 'Unknown')
    print(f'\u2713 QuickSight active (Edition: {edition})')
    if edition == 'STANDARD':
        print('  \u26a0 Definition API requires Enterprise edition')
except ClientError as e:
    if e.response['Error']['Code'] == 'ResourceNotFoundException':
        print('\u2717 QuickSight not subscribed: https://quicksight.aws.amazon.com/')
    else: raise

## 3. Verify Inference Data in Athena

In [ ]:
def run_athena_query(sql, database=ATHENA_DATABASE):
    r = athena.start_query_execution(
        QueryString=sql,
        QueryExecutionContext={'Database': database},
        ResultConfiguration={'OutputLocation': ATHENA_OUTPUT_S3},
    )
    eid = r['QueryExecutionId']
    while True:
        s = athena.get_query_execution(QueryExecutionId=eid)
        state = s['QueryExecution']['Status']['State']
        if state in ('SUCCEEDED', 'FAILED', 'CANCELLED'): break
        time.sleep(1)
    if state != 'SUCCEEDED':
        reason = s['QueryExecution']['Status'].get('StateChangeReason', '')
        raise Exception(f'Query {state}: {reason}')
    return athena.get_query_results(QueryExecutionId=eid)

print('Checking inference_responses table...')
r = run_athena_query('SELECT COUNT(*) FROM inference_responses')
print(f'  Total records: {r["ResultSet"]["Rows"][1]["Data"][0]["VarCharValue"]}')
r = run_athena_query('SELECT COUNT(*) FROM inference_responses WHERE ground_truth IS NOT NULL')
print(f'  With ground truth: {r["ResultSet"]["Rows"][1]["Data"][0]["VarCharValue"]}')
r = run_athena_query('SELECT MIN(request_timestamp), MAX(request_timestamp) FROM inference_responses')
print(f'  Range: {r["ResultSet"]["Rows"][1]["Data"][0].get("VarCharValue","N/A")} \u2192 {r["ResultSet"]["Rows"][1]["Data"][1].get("VarCharValue","N/A")}')
print('\u2713 Data available')

## 4. Create Athena Data Source in QuickSight

In [ ]:
def get_quicksight_principal():
    try:
        users = quicksight.list_users(AwsAccountId=ACCOUNT_ID, Namespace='default')
        if users['UserList']:
            return users['UserList'][0]['Arn']
    except Exception:
        pass
    return f'arn:aws:quicksight:{REGION}:{ACCOUNT_ID}:user/default/Admin/*'

QS_PRINCIPAL = get_quicksight_principal()
print(f'QuickSight principal: {QS_PRINCIPAL}')

DS_ACTIONS = [
    'quicksight:DescribeDataSource', 'quicksight:DescribeDataSourcePermissions',
    'quicksight:PassDataSource', 'quicksight:UpdateDataSource',
    'quicksight:DeleteDataSource', 'quicksight:UpdateDataSourcePermissions',
]
ds_common = dict(
    AwsAccountId=ACCOUNT_ID, DataSourceId=DATASOURCE_ID,
    Name='Fraud Governance - Athena',
    DataSourceParameters={'AthenaParameters': {'WorkGroup': 'primary'}},
)
try:
    quicksight.describe_data_source(AwsAccountId=ACCOUNT_ID, DataSourceId=DATASOURCE_ID)
    print('Updating existing data source...')
    resp = quicksight.update_data_source(**ds_common)
except ClientError as e:
    if e.response['Error']['Code'] == 'ResourceNotFoundException':
        print('Creating new data source...')
        resp = quicksight.create_data_source(
            **ds_common, Type='ATHENA',
            Permissions=[{'Principal': QS_PRINCIPAL, 'Actions': DS_ACTIONS}],
        )
    else: raise
DATASOURCE_ARN = resp['Arn']
print(f'\u2713 Data source: {DATASOURCE_ARN}')

## 5. Fix Lake Formation Permissions (if needed)

If QuickSight shows **"Lake Formation permissions missing"** when opening the analysis,
uncomment and run the commands below. This grants `IAM_ALLOWED_PRINCIPALS` access so any
IAM role with Glue/Athena permissions (including QuickSight's service role) can query the tables.

In [ ]:
# ============================================================================
# LAKE FORMATION PERMISSIONS FIX
# ============================================================================
# Uncomment and run these if QuickSight shows "Lake Formation permissions missing".
# This grants IAM_ALLOWED_PRINCIPALS access so any IAM role with Glue/Athena
# permissions (including QuickSight's service role) can query the tables.
#
# You can also run these directly in your terminal (without the leading !).

# 1. Database-level: grant ALL on fraud_detection database
# !aws lakeformation grant-permissions \
#   --principal '{"DataLakePrincipalIdentifier":"IAM_ALLOWED_PRINCIPALS"}' \
#   --resource '{"Database":{"Name":"fraud_detection"}}' \
#   --permissions "ALL" \
#   --region us-east-1

# 2. Table-level: grant on inference_responses (used by this dashboard)
# !aws lakeformation grant-permissions \
#   --principal '{"DataLakePrincipalIdentifier":"IAM_ALLOWED_PRINCIPALS"}' \
#   --resource '{"Table":{"DatabaseName":"fraud_detection","Name":"inference_responses"}}' \
#   --permissions "SELECT" "DESCRIBE" "ALTER" "DROP" "INSERT" "DELETE" \
#   --region us-east-1

# 3. Table-level: grant on ground_truth_updates
# !aws lakeformation grant-permissions \
#   --principal '{"DataLakePrincipalIdentifier":"IAM_ALLOWED_PRINCIPALS"}' \
#   --resource '{"Table":{"DatabaseName":"fraud_detection","Name":"ground_truth_updates"}}' \
#   --permissions "SELECT" "DESCRIBE" "ALTER" "DROP" "INSERT" "DELETE" \
#   --region us-east-1

# ============================================================================
# S3 PERMISSIONS FIX
# ============================================================================
# If QuickSight shows "s3:GetObject" permission denied errors, the QuickSight
# service role needs S3 read access to the data lake bucket (Iceberg metadata)
# and the Athena query results bucket.
#
# 4. Grant S3 data lake access to QuickSight service role
# !aws iam put-role-policy \
#   --role-name aws-quicksight-service-role-v0 \
#   --policy-name QuickSightS3DataLakeAccess \
#   --policy-document '{"Version":"2012-10-17","Statement":[{"Effect":"Allow","Action":["s3:GetObject","s3:ListBucket","s3:GetBucketLocation"],"Resource":["arn:aws:s3:::fraud-detection-data-lake-skoppar-*","arn:aws:s3:::fraud-detection-data-lake-skoppar-*/*"]}]}'
#
# 5. Grant Athena query results access to QuickSight service role
# !aws iam put-role-policy \
#   --role-name aws-quicksight-service-role-v0 \
#   --policy-name QuickSightAthenaResultsAccess \
#   --policy-document '{"Version":"2012-10-17","Statement":[{"Effect":"Allow","Action":["s3:GetObject","s3:PutObject","s3:ListBucket","s3:GetBucketLocation"],"Resource":["arn:aws:s3:::aws-athena-query-results-*","arn:aws:s3:::aws-athena-query-results-*/*"]}]}'

print('Permissions fix cell \u2014 uncomment commands above if needed')

## 6. Create Dataset from Inference Responses

Uses `RelationalTable` so QuickSight auto-discovers columns from Athena.
Calculated fields (`prediction_accuracy`, `risk_tier`) are added via `LogicalTableMap`.

In [ ]:
# RelationalTable — no column list needed, QuickSight auto-discovers from Athena
physical_table_map = {
    'inference-responses': {
        'RelationalTable': {
            'DataSourceArn': DATASOURCE_ARN,
            'Catalog': 'AwsDataCatalog',
            'Schema': ATHENA_DATABASE,
            'Name': 'inference_responses',
            'InputColumns': [
                {'Name': 'inference_id', 'Type': 'STRING'},
                {'Name': 'request_timestamp', 'Type': 'DATETIME'},
                {'Name': 'endpoint_name', 'Type': 'STRING'},
                {'Name': 'model_version', 'Type': 'STRING'},
                {'Name': 'mlflow_run_id', 'Type': 'STRING'},
                {'Name': 'input_features', 'Type': 'STRING'},
                {'Name': 'prediction', 'Type': 'INTEGER'},
                {'Name': 'probability_fraud', 'Type': 'DECIMAL'},
                {'Name': 'probability_non_fraud', 'Type': 'DECIMAL'},
                {'Name': 'confidence_score', 'Type': 'DECIMAL'},
                {'Name': 'ground_truth', 'Type': 'INTEGER'},
                {'Name': 'ground_truth_timestamp', 'Type': 'DATETIME'},
                {'Name': 'ground_truth_source', 'Type': 'STRING'},
                {'Name': 'days_to_ground_truth', 'Type': 'DECIMAL'},
                {'Name': 'inference_latency_ms', 'Type': 'DECIMAL'},
                {'Name': 'model_load_time_ms', 'Type': 'DECIMAL'},
                {'Name': 'preprocessing_time_ms', 'Type': 'DECIMAL'},
                {'Name': 'transaction_id', 'Type': 'STRING'},
                {'Name': 'transaction_amount', 'Type': 'DECIMAL'},
                {'Name': 'customer_id', 'Type': 'STRING'},
                {'Name': 'is_high_confidence', 'Type': 'BIT'},
                {'Name': 'is_low_confidence', 'Type': 'BIT'},
                {'Name': 'prediction_bucket', 'Type': 'STRING'},
                {'Name': 'request_id', 'Type': 'STRING'},
                {'Name': 'response_time', 'Type': 'DATETIME'},
                {'Name': 'error_message', 'Type': 'STRING'},
                {'Name': 'inference_mode', 'Type': 'STRING'},
            ],
        }
    }
}

# Calculated fields via LogicalTableMap
logical_table_map = {
    'inference-responses-logical': {
        'Alias': 'Inference Responses',
        'Source': {'PhysicalTableId': 'inference-responses'},
        'DataTransforms': [
            {
                'CreateColumnsOperation': {
                    'Columns': [
                        {
                            'ColumnName': 'prediction_accuracy',
                            'ColumnId': 'prediction-accuracy',
                            'Expression': (
                                "ifelse("
                                "isNull({ground_truth}), 'Pending', "
                                "ifelse({prediction} = {ground_truth}, 'Correct', 'Incorrect'))"
                            ),
                        },
                        {
                            'ColumnName': 'risk_tier',
                            'ColumnId': 'risk-tier',
                            'Expression': (
                                "ifelse("
                                "{probability_fraud} > 0.8, 'High Risk', "
                                "ifelse({probability_fraud} > 0.5, 'Medium Risk', "
                                "ifelse({probability_fraud} > 0.2, 'Low Risk', 'Minimal Risk')))"
                            ),
                        },
                    ]
                }
            }
        ],
    }
}

DSET_ACTIONS = [
    'quicksight:DescribeDataSet', 'quicksight:DescribeDataSetPermissions',
    'quicksight:PassDataSet', 'quicksight:DescribeIngestion',
    'quicksight:ListIngestions', 'quicksight:UpdateDataSet',
    'quicksight:DeleteDataSet', 'quicksight:CreateIngestion',
    'quicksight:CancelIngestion', 'quicksight:UpdateDataSetPermissions',
]
dset_common = dict(
    AwsAccountId=ACCOUNT_ID, DataSetId=DATASET_ID,
    Name='Fraud Governance - Inference Monitoring',
    PhysicalTableMap=physical_table_map,
    LogicalTableMap=logical_table_map,
    ImportMode='DIRECT_QUERY',
)
try:
    quicksight.describe_data_set(AwsAccountId=ACCOUNT_ID, DataSetId=DATASET_ID)
    print('Updating existing dataset...')
    resp = quicksight.update_data_set(**dset_common)
except ClientError as e:
    if e.response['Error']['Code'] == 'ResourceNotFoundException':
        print('Creating new dataset...')
        resp = quicksight.create_data_set(
            **dset_common,
            Permissions=[{'Principal': QS_PRINCIPAL, 'Actions': DSET_ACTIONS}],
        )
    else: raise
DATASET_ARN = resp['Arn']
print(f'\u2713 Dataset: {DATASET_ARN}')

## 6. Define Visuals

Six visuals for the governance analysis:
1. Prediction Volume Over Time (line chart)
2. Fraud Probability Distribution (histogram)
3. Prediction Accuracy Breakdown (donut)
4. Risk Tier Distribution (bar chart)
5. Inference Latency Trend (line chart)
6. Total Inferences KPI

In [ ]:
# Helper: column identifier shorthand
def col(name):
    return {'DataSetIdentifier': 'inference-ds', 'ColumnName': name}

# V1: Prediction Volume Over Time (line chart)
v1_volume = {
    'LineChartVisual': {
        'VisualId': 'v1-prediction-volume',
        'Title': {'Visibility': 'VISIBLE', 'FormatText': {'PlainText': 'Prediction Volume Over Time'}},
        'ChartConfiguration': {
            'FieldWells': {
                'LineChartAggregatedFieldWells': {
                    'Category': [{'DateDimensionField': {'FieldId': 'date-dim', 'Column': col('request_timestamp'), 'DateGranularity': 'DAY'}}],
                    'Values': [{'NumericalMeasureField': {'FieldId': 'count-id', 'Column': col('inference_id'), 'AggregationFunction': {'SimpleNumericalAggregation': 'COUNT'}}}],
                    'Colors': [],
                }
            }
        }
    }
}

# V2: Fraud Probability Distribution (histogram)
v2_histogram = {
    'HistogramVisual': {
        'VisualId': 'v2-fraud-prob-dist',
        'Title': {'Visibility': 'VISIBLE', 'FormatText': {'PlainText': 'Fraud Probability Distribution'}},
        'ChartConfiguration': {
            'FieldWells': {
                'HistogramAggregatedFieldWells': {
                    'Values': [{'NumericalMeasureField': {'FieldId': 'prob-fraud', 'Column': col('probability_fraud')}}],
                }
            },
            'BinOptions': {'BinCount': {'Value': 20}},
        }
    }
}

# V3: Prediction Accuracy (pie/donut chart)
v3_accuracy = {
    'PieChartVisual': {
        'VisualId': 'v3-prediction-accuracy',
        'Title': {'Visibility': 'VISIBLE', 'FormatText': {'PlainText': 'Prediction Accuracy Breakdown'}},
        'ChartConfiguration': {
            'FieldWells': {
                'PieChartAggregatedFieldWells': {
                    'Category': [{'CategoricalDimensionField': {'FieldId': 'acc-cat', 'Column': col('prediction_accuracy')}}],
                    'Values': [{'NumericalMeasureField': {'FieldId': 'acc-count', 'Column': col('inference_id'), 'AggregationFunction': {'SimpleNumericalAggregation': 'COUNT'}}}],
                }
            },
            'DonutOptions': {'ArcOptions': {'ArcThickness': 'MEDIUM'}},
        }
    }
}

# V4: Risk Tier Distribution (bar chart)
v4_risk = {
    'BarChartVisual': {
        'VisualId': 'v4-risk-tier',
        'Title': {'Visibility': 'VISIBLE', 'FormatText': {'PlainText': 'Risk Tier Distribution'}},
        'ChartConfiguration': {
            'FieldWells': {
                'BarChartAggregatedFieldWells': {
                    'Category': [{'CategoricalDimensionField': {'FieldId': 'risk-cat', 'Column': col('risk_tier')}}],
                    'Values': [{'NumericalMeasureField': {'FieldId': 'risk-count', 'Column': col('inference_id'), 'AggregationFunction': {'SimpleNumericalAggregation': 'COUNT'}}}],
                    'Colors': [],
                }
            },
            'Orientation': 'VERTICAL',
        }
    }
}

# V5: Inference Latency Trend (line chart) — uses inference_latency_ms
v5_latency = {
    'LineChartVisual': {
        'VisualId': 'v5-latency-trend',
        'Title': {'Visibility': 'VISIBLE', 'FormatText': {'PlainText': 'Inference Latency Trend (ms)'}},
        'ChartConfiguration': {
            'FieldWells': {
                'LineChartAggregatedFieldWells': {
                    'Category': [{'DateDimensionField': {'FieldId': 'lat-date', 'Column': col('request_timestamp'), 'DateGranularity': 'DAY'}}],
                    'Values': [{'NumericalMeasureField': {'FieldId': 'avg-latency', 'Column': col('inference_latency_ms'), 'AggregationFunction': {'SimpleNumericalAggregation': 'AVERAGE'}}}],
                    'Colors': [],
                }
            }
        }
    }
}

# V6: KPI — Total Inferences
v6_kpi = {
    'KPIVisual': {
        'VisualId': 'v6-total-inferences',
        'Title': {'Visibility': 'VISIBLE', 'FormatText': {'PlainText': 'Total Inferences'}},
        'ChartConfiguration': {
            'FieldWells': {
                'Values': [{'NumericalMeasureField': {'FieldId': 'kpi-count', 'Column': col('inference_id'), 'AggregationFunction': {'SimpleNumericalAggregation': 'COUNT'}}}],
            }
        }
    }
}

VISUALS = [v1_volume, v2_histogram, v3_accuracy, v4_risk, v5_latency, v6_kpi]
print(f'Defined {len(VISUALS)} visuals for the analysis')

## 7. Create Analysis via Definition API

In [ ]:
ANALYSIS_ACTIONS = [
    'quicksight:RestoreAnalysis', 'quicksight:UpdateAnalysisPermissions',
    'quicksight:DeleteAnalysis', 'quicksight:DescribeAnalysisPermissions',
    'quicksight:QueryAnalysis', 'quicksight:DescribeAnalysis', 'quicksight:UpdateAnalysis',
]

analysis_definition = {
    'DataSetIdentifierDeclarations': [{
        'Identifier': 'inference-ds',
        'DataSetArn': DATASET_ARN,
    }],
    'Sheets': [{
        'SheetId': 'governance-sheet-1',
        'Name': 'Inference Monitoring',
        'Visuals': VISUALS,
    }],
    'AnalysisDefaults': {
        'DefaultNewSheetConfiguration': {
            'InteractiveLayoutConfiguration': {
                'FreeForm': {'CanvasSizeOptions': {'ScreenCanvasSizeOptions': {'OptimizedViewPortWidth': '1600px'}}}
            }
        }
    },
}

try:
    quicksight.describe_analysis(AwsAccountId=ACCOUNT_ID, AnalysisId=ANALYSIS_ID)
    print('Analysis exists, updating...')
    resp = quicksight.update_analysis(
        AwsAccountId=ACCOUNT_ID, AnalysisId=ANALYSIS_ID,
        Name='Fraud Detection Governance Analysis',
        Definition=analysis_definition,
    )
except ClientError as e:
    if e.response['Error']['Code'] == 'ResourceNotFoundException':
        print('Creating analysis with Definition mode...')
        resp = quicksight.create_analysis(
            AwsAccountId=ACCOUNT_ID, AnalysisId=ANALYSIS_ID,
            Name='Fraud Detection Governance Analysis',
            Definition=analysis_definition,
            Permissions=[{'Principal': QS_PRINCIPAL, 'Actions': ANALYSIS_ACTIONS}],
        )
    else: raise

ANALYSIS_ARN = resp['Arn']
print(f'\u2713 Analysis created: {ANALYSIS_ARN}')
print(f'  Open in QuickSight: https://{REGION}.quicksight.aws.amazon.com/sn/analyses/{ANALYSIS_ID}')

## 8. Publish Dashboard via Definition API

In [ ]:
DASHBOARD_ACTIONS = [
    'quicksight:DescribeDashboard', 'quicksight:ListDashboardVersions',
    'quicksight:UpdateDashboardPermissions', 'quicksight:QueryDashboard',
    'quicksight:UpdateDashboard', 'quicksight:DeleteDashboard',
    'quicksight:DescribeDashboardPermissions', 'quicksight:UpdateDashboardPublishedVersion',
]

dashboard_definition = {
    'DataSetIdentifierDeclarations': [{
        'Identifier': 'inference-ds',
        'DataSetArn': DATASET_ARN,
    }],
    'Sheets': [{
        'SheetId': 'governance-sheet-1',
        'Name': 'Inference Monitoring',
        'Visuals': VISUALS,
    }],
}

publish_options = {
    'AdHocFilteringOption': {'AvailabilityStatus': 'ENABLED'},
    'ExportToCSVOption': {'AvailabilityStatus': 'ENABLED'},
    'SheetControlsOption': {'VisibilityState': 'EXPANDED'},
}

try:
    quicksight.describe_dashboard(AwsAccountId=ACCOUNT_ID, DashboardId=DASHBOARD_ID)
    print('Dashboard exists, updating...')
    resp = quicksight.update_dashboard(
        AwsAccountId=ACCOUNT_ID, DashboardId=DASHBOARD_ID,
        Name='Fraud Detection Governance',
        Definition=dashboard_definition,
        DashboardPublishOptions=publish_options,
    )
    version = resp['VersionArn'].split('/')[-1]
    quicksight.update_dashboard_published_version(
        AwsAccountId=ACCOUNT_ID, DashboardId=DASHBOARD_ID, VersionNumber=int(version)
    )
    print(f'  Published version {version}')
except ClientError as e:
    if e.response['Error']['Code'] == 'ResourceNotFoundException':
        print('Creating dashboard with Definition mode...')
        resp = quicksight.create_dashboard(
            AwsAccountId=ACCOUNT_ID, DashboardId=DASHBOARD_ID,
            Name='Fraud Detection Governance',
            Definition=dashboard_definition,
            DashboardPublishOptions=publish_options,
            Permissions=[{'Principal': QS_PRINCIPAL, 'Actions': DASHBOARD_ACTIONS}],
        )
    else: raise

dashboard_url = f'https://{REGION}.quicksight.aws.amazon.com/sn/dashboards/{DASHBOARD_ID}'
print(f'\u2713 Dashboard published: {dashboard_url}')

## 9. Generate Embed URL (Optional)

In [ ]:
try:
    resp = quicksight.get_dashboard_embed_url(
        AwsAccountId=ACCOUNT_ID, DashboardId=DASHBOARD_ID,
        IdentityType='QUICKSIGHT', SessionLifetimeInMinutes=600,
        UndoRedoDisabled=False, ResetDisabled=False,
    )
    print(f'Embed URL (valid 10 hours):\n  {resp["EmbedUrl"]}')
    print(f'\nHTML:\n  <iframe src="{resp["EmbedUrl"]}" width="100%" height="800px"></iframe>')
except ClientError as e:
    print(f'Could not generate embed URL: {e.response["Error"]["Message"]}')
    print('Ensure embedding is enabled in QuickSight admin settings.')

## 10. Cleanup (Optional)

Uncomment and run to delete all QuickSight resources created by this notebook.

In [ ]:
# CONFIRM_DELETE = False  # Set True to delete
#
# if CONFIRM_DELETE:
#     for res_type, res_id, id_key in [
#         ('dashboard', DASHBOARD_ID, 'DashboardId'),
#         ('analysis', ANALYSIS_ID, 'AnalysisId'),
#         ('data_set', DATASET_ID, 'DataSetId'),
#         ('data_source', DATASOURCE_ID, 'DataSourceId'),
#     ]:
#         try:
#             getattr(quicksight, f'delete_{res_type}')(AwsAccountId=ACCOUNT_ID, **{id_key: res_id})
#             print(f'  \u2713 Deleted {res_type}: {res_id}')
#         except ClientError as e:
#             if e.response['Error']['Code'] != 'ResourceNotFoundException':
#                 print(f'  \u26a0 {res_type}: {e}')
#     print('\u2713 Cleanup complete')

print('Cleanup cell \u2014 uncomment and set CONFIRM_DELETE = True to run')